
# 创建项目目录 
- 项目根目录 名称： quant-mlops-project
- 创建子目录  mkdir -p src/data data/raw notebooks
- 注意：以下记录内容，如果没有特别说明，所有命令都是在根目录下执行

# 使用uv作为python虚拟环境
- 安装   sudo UV_INSTALL_DIR=/usr/local/bin sh -c 'curl -LsSf https://astral.sh/uv/install.sh | sh'
- 查看版本信息  uv --version

# 配置uv环境
- uv init --python 3.11

# 安装python 到当前uv环境的，uv 管理的所有 Python 都在 ~/.local/share/uv/python/ 下，一个版本一个目录。这是 uv 的私有仓库。
- uv python install 3.11  
# 验证结果，并生成.venv文件
- uv run python --version

# 激活uv 环境
- source .venv/bin/activate

# 安装依赖库
- uv add jupyter
- uv add yfinance pandas pyarrow 


# 在 uv 项目里给 Jupyter 加 kernel
- uv add ipykernel 
- uv run python -m ipykernel install --user --name=quant-mlops --display-name="Python (quant-mlops)"

# 数据下载
0. 写 src/data/tickers.py
- 详见 quant-mlops-project/src/data/tickers.py   一共有100只大盘股
2. 写 src/data/download.py
- 具体见：quant-mlops-project/src/data/download.py
3.执行数据下载 uv run python src/data/download.py

In [5]:
# 检查数据情况
! uv run python -c "import pandas as pd; df = pd.read_parquet('../data/raw/AAPL.parquet'); print(df.head()); print(df.dtypes); print(df.shape)"

        date      close       high        low       open     volume ticker
0 2015-01-02  24.171757  24.638257  23.733998  24.627201  212818400   AAPL
1 2015-01-05  23.490795  24.021411  23.305080  23.941819  257142000   AAPL
2 2015-01-06  23.493011  23.751686  23.132634  23.554916  263188400   AAPL
3 2015-01-07  23.822439  23.921929  23.590294  23.700839  160423600   AAPL
4 2015-01-08  24.737745  24.795229  24.032468  24.149647  237458000   AAPL
date      datetime64[ms]
close            float64
high             float64
low              float64
open             float64
volume             int64
ticker               str
dtype: object
(2952, 7)


# 数据校验 + 处理层
- 目标：把 100 个 raw parquet 合成一张干净的、可校验的、带元数据的表。这一步决定后面特征能不能做对。

1. 写 src/data/validate.py
- 详见：quant-mlops-project/src/data/validate.py
- 检查原始数据质量，是否符合要求

In [17]:
import os
print(os.getcwd())

/home/mlops/quant-mlops-project/notebooks


In [1]:
# 跑验证数据的脚本：notebook中的脚本
import subprocess

result = subprocess.run(
    ["uv", "run", "python", "/home/mlops/quant-mlops-project/src/data/validate.py"],
    cwd="/home/mlops/quant-mlops-project",
    capture_output=True, text=True
)

print("returncode:", result.returncode)
print("stdout:")
print(result.stdout)
print("stderr:")
print(result.stderr)

returncode: 0
stdout:
[OK] AAPL: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ABBV: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ABT: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ACN: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ADBE: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ADI: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] ADP: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AMAT: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AMD: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AMGN: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AMZN: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AVGO: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] AXP: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BA: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BAC: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BKNG: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BLK: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BMY: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BRK-B: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] BSX: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] C: 2952 rows, 2015-01-02 ~ 2026-09-29
[OK] CAT: 2952 rows, 2015-

In [ ]:
# 在终端中测试的命令：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python -m src.data.validate 

2. 写 src/data/build_processed.py，连接10张表，并做数据质量校验
- 详见  quant-mlops-project/src/data/build_processed.py
- 在终端中，执行脚本：

(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python src/data/build_processed.py

found 100 tickers
saved 293978 rows -> data/processed/prices.parquet
date range: 2015-01-02 ~ 2026-09-29
tickers: 100

In [2]:
import pandas as pd
df = pd.read_parquet('/home/mlops/quant-mlops-project/data/processed/prices.parquet')
df.describe()

,date,open,high,low,close,volume
count,293978,293978.000000,293978.000000,293978.000000,293978.000000,2.939780e+05
mean,2020-11-18 16:53:18.410000,152.470551,154.121585,150.779280,152.486611,1.731396e+07
min,2015-01-02 00:00:00,0.462494,0.466803,0.453397,0.458185,0.000000e+00
25%,2017-12-13 00:00:00,51.645027,52.170843,51.101987,51.649116,2.534300e+06
50%,2020-11-19 00:00:00,98.033995,99.023865,97.000000,98.024895,5.435400e+06
75%,2023-10-24 00:00:00,193.446117,195.502158,191.329535,193.456882,1.295878e+07
max,2026-09-29 00:00:00,1274.130005,1292.650024,1243.229980,1280.339966,3.692928e+09
std,NaN,156.614679,158.379446,154.769258,156.608263,5.482312e+07


In [3]:
df.head()

,date,ticker,open,high,low,close,volume
0,2015-01-02,AAPL,24.627197,24.638253,23.733994,24.171753,212818400
1,2015-01-02,ABBV,40.838411,41.437507,40.838411,41.119236,5086100
2,2015-01-02,ABT,36.032918,36.192180,35.547170,35.754211,3216600
3,2015-01-02,ACN,73.433128,73.777076,72.417661,72.753418,2021300
4,2015-01-02,ADBE,72.699997,73.199997,71.889999,72.339996,2349200


In [4]:
df.tail()

,date,ticker,open,high,low,close,volume
293973,2026-09-29,VZ,45.848795,46.193447,45.277653,45.277653,19114200
293974,2026-09-29,WFC,80.620003,81.129997,80.120003,80.480003,12884500
293975,2026-09-29,WMT,107.750000,107.900002,105.349998,106.800003,27274900
293976,2026-09-29,XOM,160.220001,161.970001,159.270004,161.350006,9160000
293977,2026-09-29,ZTS,71.400002,71.699997,70.000000,70.300003,3520900
